## 🔥 <span style='text-decoration: double underline; color:rgb(10,110,217)'>**PyTorch: Fundamentals**</span>

**Author:** Ángel Fernández Caravaca

**GitHub:** [@afdezcaravaca](https://github.com/afdezcaravaca)

**Date:** March 2026

### 📑 <span style='color:rgb(10,110,217)'><u>**Introduction to PyTorch**</u></span>

#### ❓<span style='color:rgb(10,110,217)'><u>**What is PyTorch?**</u></span>
PyTorch is an open-source deep learning framework originally developed by Meta's AI Research lab (FAIR). It has become the dominant tool in both academic research and production environments, thanks to its **Python-first design philosophy**, its flexibility for rapid prototyping, and its performance for large-scale training.

At its core, PyTorch is built around two foundational abstractions:

- **`torch.Tensor`** — the primary data structure, functionally equivalent to NumPy's `ndarray` but engineered for hardware-accelerated parallel computation (NVIDIA GPUs via CUDA, Apple Silicon via MPS).
- **`torch.autograd`** — an automatic differentiation engine that tracks all tensor operations and computes gradients through backpropagation, which is the mathematical backbone of training neural networks.

#### 🌊 <span style='color:rgb(10,110,217)'><u>**The Architectural Advantage: Dynamic Computational Graphs**</u></span>

The most defining architectural feature of PyTorch is its implementation of **Dynamic Computational Graphs** (also known as the *Define-by-Run* paradigm).

Unlike older frameworks (e.g., early TensorFlow 1.x) that required constructing a static, pre-compiled computation graph before any data flowed through it, PyTorch builds the graph **on the fly**, operation by operation, as Python code executes.

This design offers critical engineering advantages:
<div align='center'>

| Feature | Static Graph | Dynamic Graph (PyTorch) |
|---|---|---|
| Debugging | Requires special tools | Native Python debuggers (`pdb`, `print`) |
| Control flow | Must be encoded in the graph DSL | Standard Python `if`, `for`, `while` |
| Interoperability | Often isolated | Seamless with NumPy, Pandas, SciPy |
| Flexibility | Limited | Ideal for variable-length sequences, recursive models |
</div>

* **Native Debugging:** Because execution is immediate, developers can use standard Python debuggers (like `pdb`) and standard `print` statements to inspect tensor values and gradients at any arbitrary step.
* **Control Flow Flexibility:** It natively supports standard Python control flow (`if`, `for`, `while`) within the model's forward pass, making it structurally ideal for complex architectures like recurrent networks with variable-length sequences or recursive structures.
* **Ecosystem Interoperability:** It avoids creating an isolated sub-language, ensuring seamless zero-copy interoperability with libraries like NumPy, Pandas, and SciPy.


#### 🧱 <span style='color:rgb(10,110,217)'><u>**Tensors and Ranks:**</u></span>
A **tensor** is a generalization of scalars, vectors, and matrices to an arbitrary number of dimensions. In PyTorch, every piece of data is represented as a `torch.Tensor`. The **rank** of a tensor refers to its number of dimensions (`ndim`):

- **Rank 0 Tensor** $\rightarrow$ () $\rightarrow$ Dimensionless, a mathematical scalar.
- **Rank 1 Tensor** $\rightarrow$ (N,) $\rightarrow$ A 1D flat array/vector, without implicit row/column orientation.
- **Rank 2 Tensor** $\rightarrow$ (H, W) or (Batch, Features) $\rightarrow$ A 2D matrix, including explicit (1, N) row and (N, 1) column vectors.
- **Rank 3 Tensor** $\rightarrow$ (C, H, W) $\rightarrow$ A 3D tensor, such as a multi-channel spatial matrix like a single image, or a temporal sequence of embeddings.
- **Rank 4 Tensor** $\rightarrow$ (B, C, H, W) $\rightarrow$ A 4D tensor, which usually represents a batched collection of 3D tensors, which is the standard input requirement for most neural network layers.

<div align='center'>
<img src="https://cdn-images-1.medium.com/max/2000/1*_D5ZvufDS38WkhK9rK32hQ.jpeg" alt="Tensors" width="800">

</div>

Understanding tensor rank is critical because most neural network layers impose strict dimensional requirements. For instance, `Conv2d` always expects input of rank 4.

#### 🎫 <span style='color:rgb(10,110,217)'><u>**Data types**</u></span>

PyTorch supports a rich type system for tensors. The **numeric suffix** (8, 16, 32, 64) denotes the number of bits used to store each value:

- **Larger bit-width** → greater precision or range, but higher memory consumption and slower computation.
- **Smaller bit-width** → less memory, faster throughput (especially on GPUs), at the cost of numerical precision.

The **`u` prefix** (e.g., `uint8`) denotes **unsigned** types — restricted to zero and positive integers, which doubles the representable positive range relative to their signed counterpart.

**PyTorch defaults:** `float32` for floating-point tensors, `int64` for integer tensors. NumPy, by contrast, defaults to `float64`, which can cause silent precision mismatches when interoperating between the two.

Common types:

<div align='center'>

| Type | Bits | Use case |
|---|---|---|
| `torch.bool` | 1 | Boolean masks |
| `torch.int8 / uint8` | 8 | Quantized models, image pixel values |
| `torch.float16 / bfloat16` | 16 | Mixed-precision training |
| `torch.float32` | 32 | Default for NN weights |
| `torch.float64` | 64 | High-precision scientific computation |
</div>

Type casting is performed via `.to(dtype=...)` or convenience methods like `.float()`, `.int()`, `.bool()`.


#### 🔪 <span style='color:rgb(10,110,217)'><u>**Tensor Indexing and Slicing**</u></span>

PyTorch follows NumPy-style indexing conventions. Tensors can be accessed using standard Python slice notation `[start:stop:step]` along any dimension.

Key concepts:

- **Row selection:** `tensor[i, :]` selects the entire i-th row.
- **Column selection:** `tensor[:, j]` selects the entire j-th column.
- **Sub-tensor extraction:** `tensor[-2:, -2:]` extracts the bottom-right 2×2 block using negative indexing.
- **Boolean masking:** A boolean tensor of the same shape can be used to filter elements satisfying a condition (`tensor[tensor > 5]`).
- **`torch.where(condition)`:** Returns the indices of elements satisfying a condition, analogous to `np.where`.
- **`torch.masked_select(tensor, mask)`:** Extracts a flat 1D tensor of all elements where the mask is `True`.


#### 📐<span style='color:rgb(10,110,217)'><u>**Shape Manipulation**</u></span>

Reshaping is a fundamental operation in deep learning pipelines. PyTorch provides several tools:

##### `reshape` / `view`
Both change the logical shape of a tensor without modifying the underlying data. The total number of elements must be preserved.

- **`.view()`** requires the tensor to be **contiguous** in memory (i.e., its logical layout matches its physical memory layout).
- **`.reshape()`** is more permissive — it calls `.contiguous()` internally if needed, making it safer for general use.

##### `squeeze` / `unsqueeze`
These operations add or remove dimensions of size 1 without changing the data volume.

- **`.unsqueeze(dim)`** — injects a new axis at position `dim`.
- **`.squeeze(dim)`** — removes the axis at position `dim` if its size is 1.

Their primary use is ensuring **dimensional compatibility** with neural network layers and broadcasting rules. For example, a single image of shape `(C, H, W)` must be unsqueezed to `(1, C, H, W)` before being passed to a `Conv2d` layer.

##### `permute` / `transpose`
- **`.permute(*dims)`** — reorders all dimensions simultaneously using their positional indices (generalization of transpose).
- **`.T`** — reverses all dimensions (only reliable and non-deprecated for 2D tensors).

> ⚠️ Both `.permute()` and `.T` on tensors with rank > 2 produce **non-contiguous** tensors (they reorder strides without moving bytes), which will cause `.view()` to raise a `RuntimeError`. The fix is `.contiguous().view(...)`.


#### 💾 <span style='color:rgb(10,110,217)'><u>**Memory: Views, Clones, and Contiguity**</u></span>

Understanding memory ownership is critical for correctness and debugging.
<div align='center'>

| Operation | Copies memory? | Shares underlying data? |
|---|---|---|
| `.view()` | No | Yes — mutations propagate |
| `.reshape()` | Sometimes | Usually yes |
| `.clone()` | Yes | No — fully independent copy |
| `.numpy()` | No | Yes — zero-copy bridge |
| `torch.from_numpy()` | No | Yes — zero-copy bridge |
| `torch.tensor(np_array)` | Yes | No — strict copy |
</div>
The **zero-copy bridge** between PyTorch and NumPy (via `.numpy()` and `torch.from_numpy()`) means that modifying the NumPy array also mutates the tensor, and vice versa. Use `.clone()` before converting when independence is required.

**Contiguity:** A tensor is contiguous when its elements are stored sequentially in memory in the same order as their logical indexing. Operations like `.T` and `.permute()` alter strides (the step size between elements in memory) without physically moving data, producing non-contiguous tensors.


#### 🔣 <span style='color:rgb(10,110,217)'><u>**Aggregation and Reduction Operations**</u></span>

PyTorch provides a rich set of aggregation functions: `.sum()`, `.mean()`, `.std()`, `.min()`, `.max()`, `.argmin()`, `.argmax()`.

A key parameter is `dim`, which specifies the axis *along which* the reduction is applied:

- For a tensor of shape `(B, C, H, W)`:
  - `tensor.sum(dim=0)` collapses the batch dimension → result shape `(C, H, W)`.
  - `tensor.mean(dim=1)` collapses the channel dimension → result shape `(B, H, W)`.

The `keepdim=True` argument preserves the reduced dimension as a size-1 axis, which is essential for **broadcasting** in subsequent operations (e.g., standardizing by subtracting the per-row mean without introducing shape mismatches).


#### 📶 <span style='color:rgb(10,110,217)'><u>**Broadcasting**</u></span>

Broadcasting is a mechanism that allows PyTorch to perform arithmetic operations on tensors of **different but compatible shapes** without explicit data replication.

Two shapes are broadcast-compatible if, aligned from the right, each pair of dimensions is either equal or one of them is 1. For example:

```
(4, 3)  +  (1, 3)  →  (4, 3)   ✓ — the (1, 3) tensor is "virtually" tiled across rows
(4, 1)  +  (1, 3)  →  (4, 3)   ✓ — outer product-like expansion
(4, 3)  +  (4,)    →  ERROR     ✗ — shapes do not align from the right
```

Broadcasting avoids explicit loops and memory duplication, enabling **vectorized code** that is both concise and computationally efficient. A practical example is generating a multiplication table as a `(10, 10)` tensor via the outer product of a `(10, 1)` column vector and a `(1, 10)` row vector.


#### 👷<span style='color:rgb(10,110,217)'><u>**Concatenation vs. Stacking**</u></span>

Both operations combine multiple tensors, but they differ fundamentally:

<div align='center'>

| Operation | Effect on shape | New dimension? |
|---|---|---|
| `torch.cat([t1, t2], dim=d)` | Grows an existing dimension | No |
| `torch.stack([t1, t2], dim=d)` | Inserts a new dimension at position `d` | Yes |
</div>

For tensors of shape `(2, 4)`:
- `torch.cat([...], dim=0)` → `(6, 4)` — stacks rows.
- `torch.cat([...], dim=1)` → `(2, 12)` — stacks columns.
- `torch.stack([...], dim=0)` → `(3, 2, 4)` — creates a new leading dimension.

`torch.cat` requires matching shapes on all dimensions except the concatenation axis. Tensors of different ranks cannot be concatenated directly — they must first be aligned using `unsqueeze`.


#### 🥷 <span style='color:rgb(10,110,217)'><u>**Sparse Tensors**</u></span>

In many real-world scenarios (natural language processing, graph neural networks, recommendation systems), data matrices are overwhelmingly populated with zeros. Storing these as dense tensors wastes memory and computation.

PyTorch supports **sparse COO tensors** (Coordinate format), which store only the indices and values of non-zero elements:

```
Dense:  [[0, 0, 0], [0, 5, 0], [0, 0, 3]]
Sparse: indices=[[1,2],[1,2]], values=[5, 3]
```

**When are sparse tensors beneficial?**

- **Memory efficiency:** Relevant only at high sparsity ratios (typically > 90% zeros), since the index storage overhead must be offset by the savings on zero values.
- **Computational speedup:** Specialized sparse operators (e.g., `torch.smm`) skip arithmetic on zero entries entirely.

#### 🌱<span style='color:rgb(10,110,217)'><u>**Reproducibility and Random Seeds**</u></span>

Training neural networks involves stochastic operations (random weight initialization, data shuffling, dropout). Without controlling randomness, results cannot be reproduced across runs, making debugging, comparison, and publication unreliable.

`torch.manual_seed(seed)` initializes the pseudo-random number generator (PRNG) to a deterministic state. Any call to a random function after setting the seed will always produce the same sequence of values.

**Why is reproducibility critical?**

1. **Debugging:** Isolate whether a training run's behavior is due to code changes or random variation.
2. **Fair comparison:** Ensure two model variants start from identical conditions.
3. **Scientific integrity:** Allow others to replicate published results exactly.

> For full reproducibility in multi-GPU environments, additional steps are required: seeding `numpy`, `random`, and setting `torch.backends.cudnn.deterministic = True`.

---

### ⚒️ <span style='color:rgb(10,110,217)'><u>**Installation**</u></span>
Create your conda/python enviroment for PyTorch and follow these [instructions](https://pytorch.org/get-started/locally/). Notice that you must:

- **Python version $\geq$ 3.10**
- **GPU or CPU:** if you want to use PyTorch on your GPU you must install it with CUDA support. To identify the maximum CUDA version your current drivers support, run nvidia-smi in the terminal and check the "CUDA Version" field in the upper right corner.

---

### 🐤 <span style='color:rgb(10,110,217)'><u>**Level 0: Baby steps**</u></span>
Use the [official documentation](https://docs.pytorch.org/docs/stable/index.html) while solving the exercises. These exercises can be solving only wiht `import torch`.

#####  <span style='color:rgb(10,110,217)'>🔢 **Exercise 0:**</span>
> **Check your PyTorch version and available CUDA devices (names).**

In [1]:
import torch 

# 0. PyTorch version:
print(f'PyTorch version: {torch.__version__}')

if torch.cuda.is_available():
    number_of_devices = torch.cuda.device_count()
    print(f'There are {number_of_devices} CUDA devices.')
    for i in range(number_of_devices):
        print(f'Device ({i}): {torch.cuda.get_device_name(i)}')
else:
    print(f'There are no available devices')

PyTorch version: 2.10.0+cu130
There are 1 CUDA devices.
Device (0): NVIDIA GeForce RTX 3050 Ti Laptop GPU


##### <span style='color:rgb(10,110,217)'>🔢**Exercise 1:**</span>
> **Create a zeros tensor of shape `(3,4)`, a ones tensor of shape `(2,2,2)`, and a tensor with random values of shape `(5,5)`.**

In [2]:
# Zeros tensor:
zeros_tensor = torch.zeros((3,4))
H,W = zeros_tensor.shape
print(f'Zeros tensor of shape {H, W} (Rank {zeros_tensor.ndim})')
print(zeros_tensor)
print('-----'*10)

# Ones Tensor
ones_tensor = torch.ones((2,2,2))
C, H, W = ones_tensor.shape
print(f'Ones tensor of shape {C,H,W} (Rank {ones_tensor.ndim})')
print(ones_tensor)
print('-----'*10)

# Random values Tensor
random_tensor = torch.rand((5,5)) # rand --> Valores uniformes entre [0,1); randn --> valores con distribución normal (media: 0, std: 1)
H,W = random_tensor.shape
print(f'Random tensor of shape {H, W} (Rank {random_tensor.ndim})')
print(random_tensor)
print('-----'*10)

Zeros tensor of shape (3, 4) (Rank 2)
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
--------------------------------------------------
Ones tensor of shape (2, 2, 2) (Rank 3)
tensor([[[1., 1.],
         [1., 1.]],

        [[1., 1.],
         [1., 1.]]])
--------------------------------------------------
Random tensor of shape (5, 5) (Rank 2)
tensor([[0.2861, 0.3519, 0.7477, 0.8301, 0.2608],
        [0.5543, 0.0464, 0.9719, 0.9535, 0.7934],
        [0.2176, 0.1497, 0.0542, 0.3513, 0.3675],
        [0.7323, 0.8206, 0.7207, 0.9941, 0.7988],
        [0.1230, 0.6500, 0.3912, 0.2492, 0.3577]])
--------------------------------------------------


##### <span style='color:rgb(10,110,217)'>🔢**Exercise 2:**</span>

> **Given to tensors `a=[1,2,3]` and `b = [4,5,6]`, compute their sum, difference, element-wise product and matrix product.**

**Note:** torch.matmul is more versatile than torch.dot. While torch.dot strictly computes the dot product of two 1D vectors, torch.matmul supports matrix multiplication and batched operations across higher-dimensional tensors (Rank 1 and above).

In [3]:
# Define a and b as tensors:
a = torch.tensor([1,2,3])
b = torch.tensor([4,5,6])

print(f'Tensors:')
print(a, b)
print('-----'*10)

# Sum:
c = torch.add(a,b) # = a + b
print(f'Sum: {c}')
print('-----'*10)

# Difference:
c = torch.sub(a,b) # = a - b
print(f'Difference: {c}')
print('-----'*10)

# Element-wise product:
c = torch.mul(a, b) # = a*b
print(f'Element-wise product: {c}')
print('-----'*10)

# Matrix product: 
c = torch.matmul(a, b) #  = a@b ; torch.matmul is superior to torch.dot
print(f'Matrix product: {c}')

Tensors:
tensor([1, 2, 3]) tensor([4, 5, 6])
--------------------------------------------------
Sum: tensor([5, 7, 9])
--------------------------------------------------
Difference: tensor([-3, -3, -3])
--------------------------------------------------
Element-wise product: tensor([ 4, 10, 18])
--------------------------------------------------
Matrix product: 32


##### <span style='color:rgb(10,110,217)'>🔢**Exercise 3:**</span>

>**Create a tensor of shape `(4,4)` with values from 1 to 16 and extract:**
>    - **The second raw.**
>    - **The third column.**
>    - **The `2x2` subtensor of the bottom-right corner.**

In [4]:
# Create the tensor:
tensor = torch.arange(1, 17).reshape(4, 4)
print(f'Original:\n', tensor)
print('-----'*10)

# Second Row
second_row = tensor[1, :]
print(f'Second raw:\n', second_row)  
print('-----'*10)

# Third Column
third_column = tensor[:, 2]
print(f'Third column:\n', third_column)
print('-----'*10)

# Subtesor:
subtensor = tensor[-2:, -2:]  
print(f'2x2 subtensor:', subtensor)

Original:
 tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12],
        [13, 14, 15, 16]])
--------------------------------------------------
Second raw:
 tensor([5, 6, 7, 8])
--------------------------------------------------
Third column:
 tensor([ 3,  7, 11, 15])
--------------------------------------------------
2x2 subtensor: tensor([[11, 12],
        [15, 16]])


##### <span style='color:rgb(10,110,217)'>🔢**Exercise 4:**</span>
>**Create a tensor of shape `(2,3,4)`, reshape it to `(6,4)`, then to `(24,)` and back to `(2,3,4)`.**

In [5]:
# Create tensor:
tensor = torch.randn((2,3,4))
C, H, W = tensor.shape
print(f'Original: Shape {C,H,W} --> (Rank {tensor.ndim})')
print(tensor)
print('-----'*10)

# Reshape (6,4)
tensor64 = torch.reshape(tensor, (6,4))
H, W = tensor64.shape
print(f'Reshape 1: Shape {H,W} --> (Rank {tensor64.ndim})')
print(tensor64)
print('-----'*10)

# Reshape (24, )
tensor24 = torch.reshape(tensor64, (24,))
H = tensor24.shape[0]
print(f'Reshape 2: Shape {H,} --> (Rank {tensor24.ndim})')
print(tensor24)
print('-----'*10)

Original: Shape (2, 3, 4) --> (Rank 3)
tensor([[[ 0.7857, -0.6520, -1.0382,  1.7333],
         [ 0.7122,  0.6091,  0.3604,  1.1149],
         [-1.1315, -1.8505, -0.1439,  1.0320]],

        [[ 1.3702, -0.1992,  0.0673, -1.2631],
         [-0.5917,  0.3701,  0.2839,  0.2437],
         [ 0.1191, -0.1460, -0.0730, -0.6370]]])
--------------------------------------------------
Reshape 1: Shape (6, 4) --> (Rank 2)
tensor([[ 0.7857, -0.6520, -1.0382,  1.7333],
        [ 0.7122,  0.6091,  0.3604,  1.1149],
        [-1.1315, -1.8505, -0.1439,  1.0320],
        [ 1.3702, -0.1992,  0.0673, -1.2631],
        [-0.5917,  0.3701,  0.2839,  0.2437],
        [ 0.1191, -0.1460, -0.0730, -0.6370]])
--------------------------------------------------
Reshape 2: Shape (24,) --> (Rank 1)
tensor([ 0.7857, -0.6520, -1.0382,  1.7333,  0.7122,  0.6091,  0.3604,  1.1149,
        -1.1315, -1.8505, -0.1439,  1.0320,  1.3702, -0.1992,  0.0673, -1.2631,
        -0.5917,  0.3701,  0.2839,  0.2437,  0.1191, -0.1460, -

In [6]:
# Back to original:
tensor = torch.reshape(tensor24, (2,3,4))
C, H, W = tensor.shape
print(f'Back to Original: Shape {C,H,W} --> (Rank {tensor.ndim})')
print(tensor)
print('-----'*10)

Back to Original: Shape (2, 3, 4) --> (Rank 3)
tensor([[[ 0.7857, -0.6520, -1.0382,  1.7333],
         [ 0.7122,  0.6091,  0.3604,  1.1149],
         [-1.1315, -1.8505, -0.1439,  1.0320]],

        [[ 1.3702, -0.1992,  0.0673, -1.2631],
         [-0.5917,  0.3701,  0.2839,  0.2437],
         [ 0.1191, -0.1460, -0.0730, -0.6370]]])
--------------------------------------------------


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 5:**</span>
>**Create a tensor from the Python list [1,2,3,4,5]. Then, create the same tensor from a NumPy array. Convert the tensor back to NumPy. Do they share memory?**

In [7]:
# Create the list:
python_list = [1,2,3,4,5]
print(f'Original Python list:')
print(f'Type: {type(python_list)}')
print(python_list)
print('-----'*10)

# Tensor from python list:
tensor = torch.tensor(python_list)
print(f'Tensor from Python list:  ')
print(f'Type: {type(tensor)}')
print(tensor)
print('-----'*10)

# Numpy array from tensor:
np_array = tensor.numpy()
print(f'Numpy array from Tensor: ')
print(f'Type: {type(np_array)}')
print(np_array)

# Back to tensor:
# tensor2 = torch.tensor(np_array)
tensor2 = torch.from_numpy(np_array)
print(f'Back to tensor:')
print(f'Type: {type(tensor2)}')
print(tensor2)
print('-----'*10)

# Memory?
print(f'Do they share memory?')
np_array[2] = 10
print((np_array == tensor2.numpy()).all())

Original Python list:
Type: <class 'list'>
[1, 2, 3, 4, 5]
--------------------------------------------------
Tensor from Python list:  
Type: <class 'torch.Tensor'>
tensor([1, 2, 3, 4, 5])
--------------------------------------------------
Numpy array from Tensor: 
Type: <class 'numpy.ndarray'>
[1 2 3 4 5]
Back to tensor:
Type: <class 'torch.Tensor'>
tensor([1, 2, 3, 4, 5])
--------------------------------------------------
Do they share memory?
True


<span style='color:rgb(24, 184, 212)'>**Comment:**</span> Note that `.numpy()` and `torch.from_numpy()` create a view of the original data, meaning both objects share the same underlying memory buffer (mutating one affects the other). Conversely, the factory function `torch.tensor()` forces a strict copy of the data into a new memory location.


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 6:**</span>
>**Create the following tensors:**

>- **A 5x5 identity matrix using `torch.eye()`.**
>- **A tensor of uniformly spaced values between 0 and 1 using `torch.linspace()`.**
>- **A tensor of logarithmically spaced values using `torch.logspace()`.**


In [8]:
# Identity matrix:
identity = torch.eye(5,5)
print('Identity matrix: \n', identity)
print('-----'*10)

# Use of torch.linspace()
linspaced = torch.linspace(0, 10, 11) # --> [0,1,2,3,4,5,6,7,8,9, 10]
print(f'Tensor from torch.linspace(): \n', linspaced)
print('-----'*10)

# Use of torch.logspace()
logspaced = torch.logspace(0, 4, 5) # --> [10**0, 10**1, 10**2, 10**3, 10**4]
print(f'Tensor from torch.logspace:\n', logspaced)
print('-----'*10)

Identity matrix: 
 tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]])
--------------------------------------------------
Tensor from torch.linspace(): 
 tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.])
--------------------------------------------------
Tensor from torch.logspace:
 tensor([1.0000e+00, 1.0000e+01, 1.0000e+02, 1.0000e+03, 1.0000e+04])
--------------------------------------------------


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 7:**</span>
>**Create an integer tensor (int32), a float tensor (float64), and a boolean tensor. Cast them between each other using .to() or .float(). What happens when you cast a boolean to an integer?**


<span style='color:rgb(161, 60, 219)'>**Question:**</span> *What do the numeric suffixes in data types (e.g., 8, 16, 32, 64) represent?*

They represent the number of bits allocated to store the data. For example, int8 means that each integer requires 8 bits (1 byte) of memory. Larger bit sizes provide a wider range of representable values (for integers) or higher mathematical precision (for floating-point numbers), but they also consume more memory space and can increase processing time.

<span style='color:rgb(161, 60, 219)'>**Question:**</span> *What is the significance of the 'u' prefix in primitive data types like uint8?*

The "u" stands for unsigned. It indicates that the data type cannot store negative numbers; it is exclusively restricted to zero and positive integers.

In [9]:
# Integer tensor:
tensor_int32 = torch.randint(0, 10, size=(3,3), dtype=torch.int32)
print(f'Integer tensor:')
print(tensor_int32)
print('-----'*10)

# Float tensor:
tensor_float64 = torch.randn((3,3), dtype=torch.float64)
print(f'Float tensor:')
print(tensor_float64)
print('-----'*10)

# Cast between each other:
    # Float to int:
float64_2_int32 = tensor_float64.to(dtype=torch.int32)
print(f'Float to integer')
print(float64_2_int32)
print('-----'*10)

    # Int to float:
int32_2_float64 = tensor_int32.to(dtype=torch.float64)
print(f'Integer to float')
print(int32_2_float64)

# Cast bool to integer:
tensor_bool = torch.randint(0,2, (3,3), dtype=torch.bool)
print(f'Bool tensor:')
print(tensor_bool)
print('-----'*10)

bool_2_int = tensor_bool.to(dtype=torch.int32)
print(f'Bool to integer:')
print(bool_2_int)
print('-----'*10)

Integer tensor:
tensor([[7, 2, 4],
        [8, 8, 0],
        [9, 5, 4]], dtype=torch.int32)
--------------------------------------------------
Float tensor:
tensor([[-0.3864, -0.7791, -0.1250],
        [ 2.0061,  1.1591, -0.5972],
        [-1.2908, -3.8352, -0.9284]], dtype=torch.float64)
--------------------------------------------------
Float to integer
tensor([[ 0,  0,  0],
        [ 2,  1,  0],
        [-1, -3,  0]], dtype=torch.int32)
--------------------------------------------------
Integer to float
tensor([[7., 2., 4.],
        [8., 8., 0.],
        [9., 5., 4.]], dtype=torch.float64)
Bool tensor:
tensor([[False,  True, False],
        [False,  True,  True],
        [False, False, False]])
--------------------------------------------------
Bool to integer:
tensor([[0, 1, 0],
        [0, 1, 1],
        [0, 0, 0]], dtype=torch.int32)
--------------------------------------------------


<span style='color:rgb(24, 184, 212)'>**Comment:**</span> Notice that previous outputs didn't explicitly show the tensor's dtype. PyTorch omits this information in the console for default types (`float32` for floating-point numbers and `int64` for integers). Now that a non-default type was explicitly specified, it is displayed. Note: Be cautious when mixing frameworks, as NumPy defaults to float64 instead of float32

##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 8:**</span>
>**Given a random tensor of shape (6, 6), compute its mean, standard deviation, minimum, maximum, and their respective indices using argmin and argmax. Then, repeat these calculations across its rows and columns (dim=0 and dim=1).**

In [10]:
# Create random tensor:
tensor = torch.randint(0, 10, (6,6), dtype=torch.float32)
print(f'Original: \n', tensor)
print('-----'*10)

# Mean and std
tensor_mean = tensor.mean()
tensor_std = tensor.std()
print(f"Mean ± std: {tensor_mean.item():.4f} ± {tensor_std.item():.4f}")
print('-----'*10)

tensor = tensor.to(dtype=torch.int16)
minimum = tensor.min()
min_index = torch.argmin(tensor) # --> Returns the first index of appearence as if it were a flattened tensor
all_min_indexes = torch.where(tensor == minimum) # --> Returns a tuple of tensors (tensor([a1, b1,c1,...], tensor([a2, b2, c2, ...]))
all_min_coords = list(zip(all_min_indexes[0].tolist(), all_min_indexes[1].tolist()))
print(f'Minimum {minimum:.2f} at {min_index}')
print(f'Other minima at {all_min_coords}')
print()

# Maximum:
maximum = tensor.max()
max_index = torch.argmax(tensor)
all_max_indexes = torch.where(tensor == maximum)
all_max_coords = list(zip(all_max_indexes[0].tolist(), all_max_indexes[1].tolist()))
print(f'Maximum {maximum:.2f} at {max_index}')
print(f'Other maxima at {all_max_coords}')
print('-----'*10)

# Minimum across rows and colums:
rows_minimum = tensor.min(dim=1).values # --> Due to the use of dim, it returns a namedtuple
row_min_index = torch.argmin(tensor, dim=1).tolist() # --> Due to the use of dim, it returns all indexes (.tolist())
print(f'Rows minima {rows_minimum.numpy()} at {row_min_index}')
print()
cols_minimum = tensor.min(dim=0).values
col_min_index = torch.argmin(tensor, dim=0).tolist() 
print(f'Cols minima {cols_minimum.numpy()} at {col_min_index}')
print('-----'*10)

# Maximum across rows and colums:
rows_maximum = tensor.max(dim=1).values
row_max_index = torch.argmax(tensor, dim=1).tolist() 
print(f'Rows maxima {rows_maximum.numpy()} at {row_max_index}')
print()
cols_maximum = tensor.max(dim=0).values
col_max_index = torch.argmax(tensor, dim=0).tolist() 
print(f'Cols maxima {cols_maximum.numpy()} at {col_max_index}')
print('-----'*10)

Original: 
 tensor([[4., 1., 9., 0., 9., 2.],
        [4., 5., 7., 3., 1., 7.],
        [9., 7., 6., 0., 7., 8.],
        [5., 1., 9., 8., 8., 6.],
        [0., 3., 3., 9., 9., 5.],
        [0., 8., 4., 0., 8., 3.]])
--------------------------------------------------
Mean ± std: 4.9444 ± 3.2066
--------------------------------------------------
Minimum 0.00 at 3
Other minima at [(0, 3), (2, 3), (4, 0), (5, 0), (5, 3)]

Maximum 9.00 at 2
Other maxima at [(0, 2), (0, 4), (2, 0), (3, 2), (4, 3), (4, 4)]
--------------------------------------------------
Rows minima [0 1 0 1 0 0] at [3, 4, 3, 1, 0, 0]

Cols minima [0 1 3 0 1 2] at [4, 0, 4, 0, 1, 0]
--------------------------------------------------
Rows maxima [9 7 9 9 9 8] at [2, 2, 0, 2, 3, 1]

Cols maxima [9 8 9 9 9 8] at [2, 5, 0, 4, 0, 2]
--------------------------------------------------


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 9:**</span>
>**Create a tensor of shape (3, 4, 5) and practice using sum(), mean(), and max() by specifying different dimensions (dim=0, dim=1, dim=2). Observe how the shape of the result changes.**

In [11]:
# Create random tensor:
tensor = torch.randint(0, 10, (3,4,5), dtype=torch.float32)
print(f'Original: \n', tensor)
print('-----'*10)

Original: 
 tensor([[[3., 8., 9., 2., 2.],
         [2., 8., 1., 8., 0.],
         [9., 4., 1., 0., 7.],
         [6., 8., 3., 5., 8.]],

        [[9., 2., 6., 5., 7.],
         [2., 1., 6., 0., 5.],
         [7., 6., 4., 9., 9.],
         [4., 8., 2., 1., 4.]],

        [[1., 8., 1., 7., 7.],
         [6., 8., 7., 6., 9.],
         [2., 2., 3., 4., 9.],
         [0., 6., 2., 0., 1.]]])
--------------------------------------------------


In [12]:
# Sum along channels (dim=0): (3,4,5) -->(4,5)
channels_sum = tensor.sum(dim=0)
print(f'Channels sum:\n', channels_sum)
# Sum along Height (dim=1): (3,4,5) --> (3,5)
height_sum = tensor.sum(dim=1)
print(f'Height sum:\n', height_sum)
# Sum along Width (dim=2): (3,4,5) --> (3,4)
width_sum = tensor.sum(dim=2)
print(f'Width sum:\n',width_sum)

Channels sum:
 tensor([[13., 18., 16., 14., 16.],
        [10., 17., 14., 14., 14.],
        [18., 12.,  8., 13., 25.],
        [10., 22.,  7.,  6., 13.]])
Height sum:
 tensor([[20., 28., 14., 15., 17.],
        [22., 17., 18., 15., 25.],
        [ 9., 24., 13., 17., 26.]])
Width sum:
 tensor([[24., 19., 21., 30.],
        [29., 14., 35., 19.],
        [24., 36., 20.,  9.]])


In [13]:
# Mean along channels (dim=0): (3,4,5) -->(4,5)
channels_mean = tensor.mean(dim=0)
print(f'Channels mean:\n', channels_mean)
# Mean along Height (dim=1): (3,4,5) --> (3,5)
height_mean = tensor.mean(dim=1)
print(f'Height mean:\n', height_mean)
# Mean along Width (dim=2): (3,4,5) --> (3,4)
width_mean = tensor.mean(dim=2)
print(f'Width mean:\n',width_mean)

Channels mean:
 tensor([[4.3333, 6.0000, 5.3333, 4.6667, 5.3333],
        [3.3333, 5.6667, 4.6667, 4.6667, 4.6667],
        [6.0000, 4.0000, 2.6667, 4.3333, 8.3333],
        [3.3333, 7.3333, 2.3333, 2.0000, 4.3333]])
Height mean:
 tensor([[5.0000, 7.0000, 3.5000, 3.7500, 4.2500],
        [5.5000, 4.2500, 4.5000, 3.7500, 6.2500],
        [2.2500, 6.0000, 3.2500, 4.2500, 6.5000]])
Width mean:
 tensor([[4.8000, 3.8000, 4.2000, 6.0000],
        [5.8000, 2.8000, 7.0000, 3.8000],
        [4.8000, 7.2000, 4.0000, 1.8000]])


In [14]:
# Max along channels (dim=0): (3,4,5) -->(4,5)
channels_max = tensor.max(dim=0).values
print(f'Channels max:\n', channels_max)
# Max along Height (dim=1): (3,4,5) --> (3,5)
height_max = tensor.max(dim=1).values
print(f'Height max:\n', height_max)
# Max along Width (dim=2): (3,4,5) --> (3,4)
width_max = tensor.max(dim=2)
print(f'Width max:\n',width_max)

Channels max:
 tensor([[9., 8., 9., 7., 7.],
        [6., 8., 7., 8., 9.],
        [9., 6., 4., 9., 9.],
        [6., 8., 3., 5., 8.]])
Height max:
 tensor([[9., 8., 9., 8., 8.],
        [9., 8., 6., 9., 9.],
        [6., 8., 7., 7., 9.]])
Width max:
 torch.return_types.max(
values=tensor([[9., 8., 9., 8.],
        [9., 6., 9., 8.],
        [8., 9., 9., 6.]]),
indices=tensor([[2, 1, 0, 1],
        [0, 2, 3, 1],
        [1, 4, 4, 1]]))


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 10:**</span>
> **Create a 2D tensor with random values and transpose it. Then, manually standarize each row so that every row has mean 0 and std 1.**

To manually standarize an element, we must compute:
$$
z = \frac{x - \mu}{\sigma}
$$

being $\mu$ the mean and $\sigma$ the standard deviation.

In [15]:
# Create a tensor and transpose it:
tensor = torch.rand(3,3)
tensor_T = tensor.T
print(f'Original: \n', tensor)
print('-----'*10)
print(f'Transpose: \n', tensor_T)
print('-----'*10)

# Standarize  rows:
rows_means = tensor_T.mean(dim=1, keepdim=True) # keepdim = False -> (3,); keepdim=True -> (3,1)
rows_std = tensor_T.std(dim=1, keepdim=True)
tensor_std = (tensor_T- rows_means)/rows_std
print(f'Standarized tensor:\n', tensor_std)

Original: 
 tensor([[0.0599, 0.4072, 0.0125],
        [0.5584, 0.8129, 0.4754],
        [0.3624, 0.4954, 0.3939]])
--------------------------------------------------
Transpose: 
 tensor([[0.0599, 0.5584, 0.3624],
        [0.4072, 0.8129, 0.4954],
        [0.0125, 0.4754, 0.3939]])
--------------------------------------------------
Standarized tensor:
 tensor([[-1.0632,  0.9217,  0.1415],
        [-0.7714,  1.1298, -0.3584],
        [-1.1389,  0.7344,  0.4045]])


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 11:**</span>
>**Given two random integer tensors with values from 1 to 10, create boolean masks for: elements greater than 5, and elements that are equal in both tensors. Use these masks to filter values via boolean indexing.**

In [16]:
# Random integer tensors:
A = torch.randint(0, 11, (3,3))
print('Original tensor A: \n', A)
print('-----'*10)
B = torch.randint(0, 11, (3,3))
print('Original tensor B:\n', B)
print('-----'*10)

# Boolean mask:
mask_A = torch.greater(A, 5)
print('Mask A > 5:\n', mask_A)
print('-----'*10)
mask_B = torch.greater(B, 5)
print('Mask B > 5:\n', mask_B)
print('-----'*10)

# Comparision:
mask_eq = (A == B) # Torch way???
print('Mask A = B:\n', mask_eq)
print('-----'*10)

# Shared values:
shared_values = torch.masked_select(A, mask_eq) # = A[mask_eq]
print('Shared values:\n', shared_values)

Original tensor A: 
 tensor([[ 7,  8,  1],
        [10,  4,  4],
        [ 8,  4,  2]])
--------------------------------------------------
Original tensor B:
 tensor([[0, 9, 6],
        [1, 9, 3],
        [8, 0, 7]])
--------------------------------------------------
Mask A > 5:
 tensor([[ True,  True, False],
        [ True, False, False],
        [ True, False, False]])
--------------------------------------------------
Mask B > 5:
 tensor([[False,  True,  True],
        [False,  True, False],
        [ True, False,  True]])
--------------------------------------------------
Mask A = B:
 tensor([[False, False, False],
        [False, False, False],
        [ True, False, False]])
--------------------------------------------------
Shared values:
 tensor([8])


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 12:**</span>
>**Create a tensor of shape (3, 1, 4, 1, 2). Remove all dimensions of size 1 using squeeze(). Then, add dimensions at different positions using unsqueeze(). What is the practical use case for this?**

The primary purpose of manipulating dimensions of size 1 via squeeze() and unsqueeze() is to guarantee strict topological alignment between your data structures and the rigid dimensional requirements of neural network layers, loss functions, and broadcasting operations. These operations add no actual data volume, but they help your data fulfill the dimensional requirements of the deep learning structures. 

For example, a Conv2d layer always expects data of shape (B, C, H, W) $\Rightarrow$ If you have only one image (C, H, W), you must use unsqueeze(0) to inject one extra dimension at the beginning, i.e., (1, C, H, W), in order to use the `Conv2D` layer.

In [17]:
tensor = torch.randint(0, 100, (3,1,4,1,2))
print('Original: \n', tensor)
print('-----'*10)

Original: 
 tensor([[[[[40, 78]],

          [[55, 65]],

          [[31, 73]],

          [[86, 43]]]],



        [[[[64, 90]],

          [[34, 43]],

          [[88, 49]],

          [[81, 94]]]],



        [[[[46, 36]],

          [[25, 22]],

          [[98, 66]],

          [[48, 98]]]]])
--------------------------------------------------


In [18]:
# Squeeze all dimensions of size 1:
squeezed_tensor = torch.squeeze(tensor, dim=[1,3]) # = tensor.squeeze(dim=[1,3])
print('Squeezed:\n', squeezed_tensor)

Squeezed:
 tensor([[[40, 78],
         [55, 65],
         [31, 73],
         [86, 43]],

        [[64, 90],
         [34, 43],
         [88, 49],
         [81, 94]],

        [[46, 36],
         [25, 22],
         [98, 66],
         [48, 98]]])


In [19]:
# Unsqueeze all dimensions of size 1:
unsqueezed_tensor = torch.unsqueeze(squeezed_tensor, dim=1).unsqueeze(dim=3) 
print('Unsqueezed:\n', unsqueezed_tensor)

Unsqueezed:
 tensor([[[[[40, 78]],

          [[55, 65]],

          [[31, 73]],

          [[86, 43]]]],



        [[[[64, 90]],

          [[34, 43]],

          [[88, 49]],

          [[81, 94]]]],



        [[[[46, 36]],

          [[25, 22]],

          [[98, 66]],

          [[48, 98]]]]])


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 13:**</span>
>**Create three tensors of shape (2, 4) and explore the differences between:**
>    - **torch.cat() along different axes.**
>    - **torch.stack() — what new dimension does it create?**

In [20]:
tensor1 = torch.randint(0, 10, (2,4))
tensor2 = torch.randint(0, 10, (2,4))
tensor3 = torch.randint(0, 10, (2,4))

print('Original 1:\n', tensor1)
print('-----'*10)

print('Original 2:\n', tensor2)
print('-----'*10)

print('Original 3:\n', tensor3)
print('-----'*10)

Original 1:
 tensor([[7, 6, 2, 8],
        [9, 6, 7, 9]])
--------------------------------------------------
Original 2:
 tensor([[9, 5, 6, 9],
        [7, 8, 2, 4]])
--------------------------------------------------
Original 3:
 tensor([[0, 8, 7, 0],
        [1, 6, 0, 0]])
--------------------------------------------------


In [21]:
cat_dim0 = torch.cat([tensor1, tensor2, tensor3], dim=0) # (2,4) --> (6,4)
print('Concat along dim = 0 --> Concat along rows:')
H,W = cat_dim0.shape
print(f'Shape {H,W}')
print(cat_dim0)
print('-----'*10)

cat_dim1 = torch.cat([tensor1, tensor2, tensor3], dim=1) # (2,4) --> (2, 12)
print('Concat along dim = 1 --> Concat along columns:')
H,W = cat_dim1.shape
print(f'Shape {H,W}')
print(cat_dim1)

Concat along dim = 0 --> Concat along rows:
Shape (6, 4)
tensor([[7, 6, 2, 8],
        [9, 6, 7, 9],
        [9, 5, 6, 9],
        [7, 8, 2, 4],
        [0, 8, 7, 0],
        [1, 6, 0, 0]])
--------------------------------------------------
Concat along dim = 1 --> Concat along columns:
Shape (2, 12)
tensor([[7, 6, 2, 8, 9, 5, 6, 9, 0, 8, 7, 0],
        [9, 6, 7, 9, 7, 8, 2, 4, 1, 6, 0, 0]])


In [22]:
stack_dim0 = torch.stack([tensor1, tensor2, tensor3], dim=0) # (2,4) --> (3,2,4)
print('Stack along dim=0 --> Adds an extra dim at the beginning:')
C, H, W = stack_dim0.shape
print(f'Shape: {C,H,W}')
print(stack_dim0)

stack_dim1 = torch.stack([tensor1, tensor2, tensor3], dim=1) # (2,4) --> (2,3,4)
print('Stack along dim=1 --> Adds an extra between the two original dims:')
C, H, W = stack_dim1.shape
print(f'Shape: {C,H,W}')
print(stack_dim1)

Stack along dim=0 --> Adds an extra dim at the beginning:
Shape: (3, 2, 4)
tensor([[[7, 6, 2, 8],
         [9, 6, 7, 9]],

        [[9, 5, 6, 9],
         [7, 8, 2, 4]],

        [[0, 8, 7, 0],
         [1, 6, 0, 0]]])
Stack along dim=1 --> Adds an extra between the two original dims:
Shape: (2, 3, 4)
tensor([[[7, 6, 2, 8],
         [9, 5, 6, 9],
         [0, 8, 7, 0]],

        [[9, 6, 7, 9],
         [7, 8, 2, 4],
         [1, 6, 0, 0]]])


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 14:**</span>
>**Create a mock image tensor of shape (3, 32, 64) (channels, height, width). Convert it to the (64, 3, 32) format using permute(). Then, practice using transpose().**

In [23]:
tensor = torch.randint(0,10, (3,32,64))
C,H,W = tensor.shape
print(f'Original Shape: {C,H,W}')

permuted_tensor = torch.permute(tensor, dims=(2,0,1)) # --> Expect the positional indexes of the dimensions.
C,H,W = permuted_tensor.shape
print(f'Permuted Shape: {C,H,W}')

transposed_tensor = tensor.T
C,H,W = transposed_tensor.shape
print(f'Transposed Shape: {C,H,W}')

Original Shape: (3, 32, 64)
Permuted Shape: (64, 3, 32)
Transposed Shape: (64, 32, 3)


/tmp/ipykernel_9892/1995679435.py:9: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4480.)
  transposed_tensor = tensor.T


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 15:**</span>
>**Without using loops, add a 1D tensor of shape (3,) to a matrix of shape (4, 3). Then, attempt to concat a (3,) tensor to a (4,) tensor — what error do you get and why? How would you fix it?(Recommendation: use the `try`flux controller)**

In [24]:
tensor3 = torch.randint(0,10, (3,))
print('Tensor (3,):\n', tensor3)
print('-----'*10)
tensor43 = torch.randint(0,10, (4,3))
print('Tensor (4,3):\n', tensor43)
print('-----'*10)
tensor4 = torch.randint(0,10, (4,))
print('Tensor (4,):\n',tensor4) 
print('-----'*10)

Tensor (3,):
 tensor([5, 9, 7])
--------------------------------------------------
Tensor (4,3):
 tensor([[2, 3, 6],
        [8, 8, 0],
        [7, 8, 1],
        [1, 1, 8]])
--------------------------------------------------
Tensor (4,):
 tensor([6, 2, 5, 7])
--------------------------------------------------


In [25]:
# Concat (3, ) and (4,3)
try:
    tensor343 = torch.cat([tensor3, tensor43]) # dim = 0 by default
except Exception as e:
    print(f'Error of dimensions: {e}')
    print(f'Notice that you are trying to concat a (3,) and a (4,3), a Rank 1 tensor and Rank 2 tensor')
print('-----'*10)

# Concat (3,) and (4, )
try:
    print('(3,) and (4, ) can be concatenated along dim=0')
    tensor34_dim0 = torch.cat([tensor3, tensor4]) 
    print('Tensor (3,) concat (4,) dim=0:\n', tensor34_dim0)
except Exception as e:
    print(f'Error of dimensions: {e}')
print('-----'*10)

# How to fix it?
try:
    tensor343 = torch.cat([tensor3.unsqueeze(0) , tensor43]) 
    print(f'How to fix it? --> For example, we could unsqueeze the tensor.')
    print('Tensor (1,3)(4,3):\n', tensor343)
except Exception as e:
    print(f'Error: {e}')

Error of dimensions: Tensors must have same number of dimensions: got 1 and 2
Notice that you are trying to concat a (3,) and a (4,3), a Rank 1 tensor and Rank 2 tensor
--------------------------------------------------
(3,) and (4, ) can be concatenated along dim=0
Tensor (3,) concat (4,) dim=0:
 tensor([5, 9, 7, 6, 2, 5, 7])
--------------------------------------------------
How to fix it? --> For example, we could unsqueeze the tensor.
Tensor (1,3)(4,3):
 tensor([[5, 9, 7],
        [2, 3, 6],
        [8, 8, 0],
        [7, 8, 1],
        [1, 1, 8]])


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 16:**</span>
>**Using broadcasting and no loops, generate the multiplication table from 1 to 10 as a (10, 10) tensor. Hint: you will need a column vector and a row vector.**

In [26]:
a = torch.arange(1, 11).unsqueeze(1)   # (10, 1) 
b = torch.arange(1, 11).unsqueeze(0)   # (1, 10) 
# Table:
print('Multiplication table:')
table = torch.matmul(a,b)
print(table)

Multiplication table:
tensor([[  1,   2,   3,   4,   5,   6,   7,   8,   9,  10],
        [  2,   4,   6,   8,  10,  12,  14,  16,  18,  20],
        [  3,   6,   9,  12,  15,  18,  21,  24,  27,  30],
        [  4,   8,  12,  16,  20,  24,  28,  32,  36,  40],
        [  5,  10,  15,  20,  25,  30,  35,  40,  45,  50],
        [  6,  12,  18,  24,  30,  36,  42,  48,  54,  60],
        [  7,  14,  21,  28,  35,  42,  49,  56,  63,  70],
        [  8,  16,  24,  32,  40,  48,  56,  64,  72,  80],
        [  9,  18,  27,  36,  45,  54,  63,  72,  81,  90],
        [ 10,  20,  30,  40,  50,  60,  70,  80,  90, 100]])


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 17:**</span>
>**Create a tensor of shape (4, 4) and derive a new tensor using .view(16), and another using .clone(). Modify the original tensor and observe which derived tensor changes. Next, attempt to use .view(16) on a non-contiguous tensor — what error is raised? Resolve it using .contiguous().**

A tensor is considered contiguous when its logical structure—how elements are indexed across dimensions—perfectly matches the sequential, flat layout of those elements in physical memory (RAM or VRAM). Conversely, a tensor becomes non-contiguous when metadata operations like `.T` or `.permute()` alter its logical shape by modifying the stride values without actually moving the underlying bytes. Consequently, reading a non-contiguous tensor forces the hardware pointer to jump across non-adjacent memory addresses, breaking the strict linear alignment required by high-efficiency restructuring methods like `.view()`.

In [27]:
tensor = torch.randint(0,10, (4,4))
tensor_view = tensor.view(16) # <--- Allows changing shapes
tensor_clone = tensor.clone()
print('Original:\n', tensor)
print('-----'*10)
print('Tensor from view: \n', tensor_view)
print('-----'*10)
print('Tensor from clone:\n', tensor_clone)
print('-----'*10)

Original:
 tensor([[5, 3, 6, 1],
        [7, 0, 2, 9],
        [8, 3, 5, 4],
        [1, 0, 8, 0]])
--------------------------------------------------
Tensor from view: 
 tensor([5, 3, 6, 1, 7, 0, 2, 9, 8, 3, 5, 4, 1, 0, 8, 0])
--------------------------------------------------
Tensor from clone:
 tensor([[5, 3, 6, 1],
        [7, 0, 2, 9],
        [8, 3, 5, 4],
        [1, 0, 8, 0]])
--------------------------------------------------


In [28]:
tensor[0:,0] = 10
print('Original Mod:\n', tensor)
print('-----'*10)
print('Tensor from view: \n', tensor_view)
print('-----'*10)
print('Tensor from clone:\n', tensor_clone)
print('-----'*10)

Original Mod:
 tensor([[10,  3,  6,  1],
        [10,  0,  2,  9],
        [10,  3,  5,  4],
        [10,  0,  8,  0]])
--------------------------------------------------
Tensor from view: 
 tensor([10,  3,  6,  1, 10,  0,  2,  9, 10,  3,  5,  4, 10,  0,  8,  0])
--------------------------------------------------
Tensor from clone:
 tensor([[5, 3, 6, 1],
        [7, 0, 2, 9],
        [8, 3, 5, 4],
        [1, 0, 8, 0]])
--------------------------------------------------


In [29]:
noncontiguous = tensor.T
try:
    print('New View:\n', noncontiguous.view(16))
except Exception as e:
    print(f'Error --> {e}')
print('-----'*10)
try:
    print('New View (fixed):\n', noncontiguous.contiguous().view(16))
except Exception as e:
    print(f'Error --> {e}') 

Error --> view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.
--------------------------------------------------
New View (fixed):
 tensor([10, 10, 10, 10,  3,  0,  3,  0,  6,  2,  5,  8,  1,  9,  4,  0])


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 18:**</span>
>**Create a 5x5 dense tensor initialized with zeros, then inject random values into 5 unique random positions. Convert this dense tensor to a sparse representation using .to_sparse(). What computational and memory advantages do sparse tensors provide?**

They are useful in situations where the data is sparse (for example in Conway's Game of Life States).

- **Memory Efficiency:** Bypasses allocating RAM for empty space by exclusively storing non-zero values and their spatial indices. (Note: This only yields benefits at extreme sparsity ratios, typically >90% zeros, due to the coordinate storage overhead).

- **Computational Speedup:** Specialized sparse operators (e.g., torch.smm) skip floating-point operations involving zeros, slashing execution time.



In [30]:
tensor = torch.zeros(5, 5)
random_row_indexes = torch.randint(0, 5, (5,))
random_col_indexes = torch.randint(0, 5, (5,))
tensor[random_row_indexes , random_col_indexes] = torch.rand(5)
print('Original:')
print(tensor)
print('-----'*10)
sparse_torch = tensor.to_sparse()
print('Sparse:')
print(sparse_torch)

Original:
tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6486, 0.0000, 0.6012],
        [0.0000, 0.6122, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.2419, 0.0000, 0.9379]])
--------------------------------------------------
Sparse:
tensor(indices=tensor([[2, 2, 3, 4, 4],
                       [2, 4, 1, 2, 4]]),
       values=tensor([0.6486, 0.6012, 0.6122, 0.2419, 0.9379]),
       size=(5, 5), nnz=5, layout=torch.sparse_coo)


##### 🔢<span style='color:rgb(10,110,217)'>**Exercise 19:**</span>

>**Generate two random tensors of shape (3, 3) without setting a seed — do they match? Now, call torch.manual_seed(42) before each generation and check again. Write a small experiment: generate 5 pairs of random tensors (with and without a fixed seed). Why is reproducibility critical when training neural networks?**

In [31]:
for i in range(5):
    tensor_fullrandom = torch.randint(0,10, (3,3))
    print('Random Tensor:\n ', tensor_fullrandom)
    print('-----'*10)

Random Tensor:
  tensor([[6, 9, 9],
        [3, 6, 9],
        [9, 0, 0]])
--------------------------------------------------
Random Tensor:
  tensor([[2, 6, 3],
        [4, 9, 3],
        [2, 2, 4]])
--------------------------------------------------
Random Tensor:
  tensor([[9, 7, 1],
        [6, 5, 1],
        [0, 9, 7]])
--------------------------------------------------
Random Tensor:
  tensor([[8, 0, 1],
        [6, 8, 6],
        [4, 3, 4]])
--------------------------------------------------
Random Tensor:
  tensor([[5, 9, 5],
        [1, 9, 5],
        [3, 1, 6]])
--------------------------------------------------


In [32]:
for i in range(5):
    torch.manual_seed(42)
    tensor_42 = torch.randint(0,10, (3,3))
    print('Tensor with seed 42:\n ', tensor_42)
    print('-----'*10)

Tensor with seed 42:
  tensor([[2, 7, 6],
        [4, 6, 5],
        [0, 4, 0]])
--------------------------------------------------
Tensor with seed 42:
  tensor([[2, 7, 6],
        [4, 6, 5],
        [0, 4, 0]])
--------------------------------------------------
Tensor with seed 42:
  tensor([[2, 7, 6],
        [4, 6, 5],
        [0, 4, 0]])
--------------------------------------------------
Tensor with seed 42:
  tensor([[2, 7, 6],
        [4, 6, 5],
        [0, 4, 0]])
--------------------------------------------------
Tensor with seed 42:
  tensor([[2, 7, 6],
        [4, 6, 5],
        [0, 4, 0]])
--------------------------------------------------


##### 🔢 <span style='color:rgb(10,110,217)'>**Exercise 20:**</span>
>**Generate a tensor of shape (100, 5) simulating a dataset of 100 samples with 5 features. For each feature, compute the mean, standard deviation, minimum, and maximum. Normalize the dataset using min-max scaling relying exclusively on tensor operations (no loops).**

In [33]:
# 1. Generate tensor of shape (100, 5)
torch.manual_seed(42)
data = torch.randn(100, 5)
print('Original dataset:\n', data[:5]) # <--- Show only 5 rows
print('-----'*10)
# 2. Statistics per feature 
mean   = data.mean(dim=0)          
std    = data.std(dim=0)           
minval = data.min(dim=0).values    
maxval = data.max(dim=0).values    

print(f'Means: {mean}')
print(f'Stds: {std}')
print(f'minVals: {minval}')
print(f'maxVals: {maxval}')
print('-----'*10)
# 3. Min-Max normalization — fully vectorized, zero loops
data_norm = (data - minval) / (maxval - minval)
minimum = data_norm.min()
maximum = data_norm.max()
print(f'Norm dataset minimum: {minimum}')
print(f'Norm dataset maximum: {maximum}')

Original dataset:
 tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784],
        [-1.2345, -0.0431, -1.6047, -0.7521,  1.6487],
        [-0.3925, -1.4036, -0.7279, -0.5594, -0.7688],
        [ 0.7624,  1.6423, -0.1596, -0.4974,  0.4396],
        [-0.7581,  1.0783,  0.8008,  1.6806,  1.2791]])
--------------------------------------------------
Means: tensor([ 0.0403,  0.1950, -0.0625,  0.0008,  0.0445])
Stds: tensor([1.0335, 1.0593, 0.8403, 1.0259, 0.8871])
minVals: tensor([-2.3184, -2.5095, -1.7223, -2.4885, -2.5850])
maxVals: tensor([2.1442, 2.8340, 1.9653, 2.0487, 1.9890])
--------------------------------------------------
Norm dataset minimum: 0.0
Norm dataset maximum: 1.0


### 🎌 <span style='color:rgb(10,110,217)'><u>**Summary**</u></span>

The topics covered in this notebook constitute the essential foundation for every deep learning workflow in PyTorch. A solid understanding of tensor ranks, data types, shape manipulation, memory semantics, broadcasting, and normalization is a prerequisite for correctly building, debugging, and optimizing neural network architectures.